In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score

url = (
    "https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "course_lead_scoring_2026.csv"
)

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

df.head()

Dataset shape: (5000, 9)

Column types:
lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

Missing values:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [4]:
categorical = [
    "lead_source",
    "industry",
    "employment_status",
    "location",
]

numerical = [
    "annual_income",
    "number_of_courses_viewed",
    "interaction_count",
    "lead_score",
]

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

print("Remaining missing values:", df.isna().sum().sum())

print("\nIndustry counts:")
print(df["industry"].value_counts())



Remaining missing values: 0

Industry counts:
industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64


In [5]:
correlation = df[numerical].corr()
display(correlation)

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

print("\nCorrelations for the four answer options:")
for first, second in pairs:
    print(f"{first} and {second}: {correlation.loc[first, second]:.6f}")

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000



Correlations for the four answer options:
interaction_count and lead_score: 0.915746
number_of_courses_viewed and lead_score: 0.757204
number_of_courses_viewed and interaction_count: 0.721609
annual_income and interaction_count: 0.122842


In [6]:
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.copy()
df_val = df_val.copy()
df_test = df_test.copy()

y_train = df_train.pop("converted").to_numpy()
y_val = df_val.pop("converted").to_numpy()
y_test = df_test.pop("converted").to_numpy()

print("Training:", df_train.shape)
print("Validation:", df_val.shape)
print("Test:", df_test.shape)

Training: (3000, 8)
Validation: (1000, 8)
Test: (1000, 8)


In [7]:
print("Mutual information scores:")

for column in categorical:
    score = mutual_info_score(df_train[column], y_train)
    print(f"{column}: {round(float(score), 2):.2f}")

Mutual information scores:
lead_source: 0.03
industry: 0.00
employment_status: 0.02
location: 0.00


In [9]:
features = categorical + numerical

# Convert categories into one-hot encoded columns
train_dicts = df_train[features].to_dict(orient="records")
val_dicts = df_val[features].to_dict(orient="records")

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42,
)

model.fit(X_train, y_train)

# Predict conversion probabilities, then classify at 0.5
y_pred_proba = model.predict_proba(X_val)[:, 1]
y_pred = y_pred_proba >= 0.5

baseline_accuracy = accuracy_score(y_val, y_pred)

print("Validation accuracy:", baseline_accuracy)

Validation accuracy: 0.645


In [10]:
results_q5 = []

for removed_feature in features:
    remaining_features = [
        feature for feature in features
        if feature != removed_feature
    ]

    train_dicts = df_train[remaining_features].to_dict("records")
    val_dicts = df_val[remaining_features].to_dict("records")

    dv_without = DictVectorizer(sparse=False)
    X_train_without = dv_without.fit_transform(train_dicts)
    X_val_without = dv_without.transform(val_dicts)

    model_without = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42,
    )

    model_without.fit(X_train_without, y_train)

    probabilities = model_without.predict_proba(X_val_without)[:, 1]
    predictions = probabilities >= 0.5

    accuracy_without = accuracy_score(y_val, predictions)
    difference = baseline_accuracy - accuracy_without

    results_q5.append({
        "removed_feature": removed_feature,
        "accuracy_without": accuracy_without,
        "difference": difference,
    })

results_q5 = pd.DataFrame(results_q5)
display(results_q5)

options_q5 = results_q5[
    results_q5["removed_feature"].isin([
        "lead_source",
        "number_of_courses_viewed",
        "interaction_count",
    ])
]

display(options_q5.sort_values("difference"))

,removed_feature,accuracy_without,difference
0,lead_source,0.642,0.003
1,industry,0.645,0.000
2,employment_status,0.645,0.000
3,location,0.645,0.000
4,annual_income,0.724,-0.079
5,number_of_courses_viewed,0.643,0.002
6,interaction_count,0.601,0.044
7,lead_score,0.644,0.001


,removed_feature,accuracy_without,difference
5,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
6,interaction_count,0.601,0.044


In [12]:
# Encode all features again for Question 6
dv_q6 = DictVectorizer(sparse=False)

X_train_q6 = dv_q6.fit_transform(
    df_train[features].to_dict("records")
)
X_val_q6 = dv_q6.transform(
    df_val[features].to_dict("records")
)

results_q6 = []

for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model_q6 = LogisticRegression(
        solver="liblinear",
        C=c,
        max_iter=1000,
        random_state=42,
    )

    model_q6.fit(X_train_q6, y_train)

    probabilities = model_q6.predict_proba(X_val_q6)[:, 1]
    predictions = probabilities >= 0.5
    accuracy = accuracy_score(y_val, predictions)

    results_q6.append({
        "C": c,
        "accuracy": accuracy,
        "rounded_accuracy": round(accuracy, 3),
    })

results_q6 = pd.DataFrame(results_q6)
display(results_q6)

best = results_q6.sort_values(
    ["rounded_accuracy", "C"],
    ascending=[False, True],
).iloc[0]

,C,accuracy,rounded_accuracy
0,0.000001,0.598,0.598
1,0.000010,0.598,0.598
2,0.000100,0.613,0.613
3,0.001000,0.645,0.645
